# Data Preparation: HKD Peg Analysis
Merging the exchange rate, liquidity, and federal fund datasets into a single dataset aligned by date. Filtering from May 18, 2005 because that is when the 7.75-7.85 band was introduced. Using forward fill to fill in missing HIBOR on Saturdays. (Sundays are not in the data)

In [25]:
import pandas as pd
df_rates = pd.read_csv("../data/hkd_usd_exchange_rates.csv")

df_liquid = pd.read_csv("../data/liquidity.csv")

df_federal = pd.read_csv("../data/federal_funds.csv")

In [26]:
df_rates["date"] = pd.to_datetime(df_rates["end_of_day"])

df_federal["date"] = pd.to_datetime(df_federal["observation_date"])

df_liquid["date"] = pd.to_datetime(df_liquid["end_of_date"])

In [27]:
df_hk = pd.merge(df_rates, df_liquid, on="date", how="inner")
df_merged = pd.merge(df_hk, df_federal, on ="date", how="inner")

In [28]:
cols = ["date", "usd", "cu_weakside","cu_strongside", "hibor_fixing_1m", "hibor_overnight", "closing_balance", "DFF"]
df_master = df_merged[cols]
df_master = df_master.rename(columns={"cu_weakside":"weakside", "cu_strongside":"strongside", "hibor_fixing_1m":"hibor", "DFF":"fund_rate"})
df_master = df_master.set_index("date")

In [ ]:
# Using May 18, 2005 as a cutoff because that is when the 7.75-7.85 band was introduced
df_master = df_master[df_master.index >= '2005-05-18'].copy()

# Usinf forward fill since Saturday 1m HIBOR data is missing till 2006
df_master["hibor"] = df_master["hibor"].ffill()
df_master.to_csv("../data/master.csv")

In [30]:
print(df_master.isna().sum())

usd                0
weakside           0
strongside         0
hibor              0
hibor_overnight    0
closing_balance    0
fund_rate          0
dtype: int64


In [31]:
# Searching for outliers 3 standard deviations from the mean

for col in ["usd", "hibor", "hibor_overnight", "fund_rate", "closing_balance"]:
    mean = df_master[col].mean()
    sd = df_master[col].std()
    
    upper_bound = mean + (3 * sd)
    lower_bound = mean - (3 * sd)
    
    outliers = df_master[(df_master[col] > upper_bound) | (df_master[col] < lower_bound)]
    
    if not outliers.empty:
        print(f"\nFEATURE: {col}")
        print(f"Thresholds: > {upper_bound:.4f}  OR  < {lower_bound:.4f}")
        print(f"Count: {len(outliers)} extreme points found")
        print(outliers[col])


FEATURE: hibor
Thresholds: > 11.3554  OR  < -8.1315
Count: 1 extreme points found
date
2019-09-19    199.286
Name: hibor, dtype: float64

FEATURE: hibor_overnight
Thresholds: > 6.0728  OR  < -3.7044
Count: 1 extreme points found
date
2024-12-30    6.33
Name: hibor_overnight, dtype: float64


In [ ]:
# Outlier of 199.286% HIBOR on September 19, 2019. Most likely a typo.
# According to historical HSBC HIBOR data, it is supposed to be 1.98%.
df_master.loc["2019-09-19", "hibor"] = 1.98
df_master.to_csv("../data/master.csv")